# LLM Analysis Module
## CSE 655 - Deep Learning Final Project

### 📋 Bu Notebook Hakkında

Bu notebook, OCR ile çıkarılan metnin **LLM (Large Language Model)** kullanılarak analiz edilmesi için yapılan deneyleri içermektedir.

---

### 🎯 LLM Stratejisi

Sistem, doküman sınıflandırma modülünden gelen **güven skoruna** göre iki farklı modda çalışır:

| Güven Skoru | Mod | Açıklama |
|-------------|-----|----------|
| ≥ %70 | **Type-Specific** | Doküman türüne özgü bilgi çıkarımı |
| < %70 | **Generic** | Genel özet ve anahtar bilgiler |


### 📊 Doküman Türüne Göre Çıkarılan Alanlar

| Doküman Türü | Çıkarılan Alanlar |
|--------------|-------------------|
| Email | from, to, cc, date, subject, summary |
| Invoice | invoice_number, date, vendor, customer, amount, items,summary |
| Letter | from, to, date, subject, summary |
| Form | form_title, organization, fields, date, purpose,summary |
| Unknown | topic, summary, key_entities,summary |

> ⚠️ **Not:** Tüm alanlar opsiyoneldir. LLM sadece metinde bulabildiği alanları döndürür.

---

In [1]:

!pip install groq --quiet
print("✅ Groq kuruldu!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.6 MB/s eta 0:00:00
✅ Groq kuruldu!


In [13]:
from groq import Groq

#⚠️ API anahtarınızı buraya girin
# Groq API key: https://console.groq.com/keys adresinden alınabilir
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"

In [9]:
# AYARLAR
CONFIDENCE_THRESHOLD = 70.0

# SYSTEM MESSAGE
SYSTEM_MESSAGE = """You are a document analysis assistant.
Your task is to extract information from documents and return ONLY valid JSON.

STRICT RULES:
1. Return ONLY a valid JSON object
2. No markdown, no code fences, no explanations
3. No text before or after the JSON
4. If a field is not found, DO NOT include it in the response
5. All string values must be in the same language as the document"""

# PROMPT SABLONLARI
PROMPTS = {
    "email": """This is an EMAIL document.
Extract the information you can find from the text below.
Only include fields that you can actually find in the text.

Possible fields: from, to, cc, date, subject, and the email summary

Text:
{ocr_text}

Return only JSON:""",

    "invoice": """This is an INVOICE document.
Extract the information you can find from the text below.
Only include fields that you can actually find in the text.

Possible fields: invoice_number, date, vendor, customer, amount, items, due_date and the invoice summary

Text:
{ocr_text}

Return only JSON:""",

    "letter": """This is a LETTER document.
Extract the information you can find from the text below.
Only include fields that you can actually find in the text.

Possible fields: from, to, date, subject,and the letter summary

Text:
{ocr_text}

Return only JSON:""",

    "form": """This is a FORM document.
Extract the information you can find from the text below.
Only include fields that you can actually find in the text.

Possible fields: form_title, organization, fields, date, purpose, and the form summary

Text:
{ocr_text}

Return only JSON:""",

    "unknown": """The document type could not be determined with high confidence.
Analyze the text and provide a general summary.

Text:
{ocr_text}

Return JSON with these fields:
- topic: main topic of the document
- summary: brief summary of the content
- key_entities: important names, dates, numbers"""
}

print("Promptlar hazir!")

Promptlar hazir!


In [7]:
import json
import re

def analyze_with_llm(ocr_text, doc_type):
    """OCR metnini LLM ile analiz et"""

    prompt = PROMPTS.get(doc_type, PROMPTS["unknown"]).format(ocr_text=ocr_text)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=1024
    )

    result_text = response.choices[0].message.content

    # Markdown kod blogunu temizle
    result_text = re.sub(r'```json\\s*', '', result_text)
    result_text = re.sub(r'```\\s*', '', result_text)
    result_text = result_text.strip()

    try:
        result = json.loads(result_text)
    except:
        result = {"raw_response": result_text}

    return result

print("analyze_with_llm fonksiyonu hazir!")

analyze_with_llm fonksiyonu hazir!


In [10]:
# TEST
test_ocr = """From: Pottorff, Mary
Sent: Monday, March 15, 1999 4:20 PM
To: Keane, Denise
Cc: Goldberg, Henry
Subject: Corporate Affairs Survey

Denise,

In response to point 1 of your request for update on issues from Henry, attached is an analysis of the Corporate Affairs survey we sent to the markets last September.

<<File: CA Survey Results.doc>>

Page 133"""

print("LLM analiz ediyor...")
result = analyze_with_llm(test_ocr, "email")

print("\n" + "="*60)
print("ANALIZ SONUCU:")
print("="*60)
print(json.dumps(result, indent=2, ensure_ascii=False))
print("="*60)

LLM analiz ediyor...

ANALIZ SONUCU:
{
  "from": "Pottorff, Mary",
  "to": "Keane, Denise",
  "cc": "Goldberg, Henry",
  "date": "Monday, March 15, 1999 4:20 PM",
  "subject": "Corporate Affairs Survey",
  "summary": "Analysis of the Corporate Affairs survey sent to the markets last September is attached."
}
